# Project 3 — Morgan Fingerprint QSAR Regression

This notebook is a detailed practice project for **substructure-based QSAR modelling**. It uses RDKit Morgan/ECFP fingerprints to predict aqueous solubility (`logS`) from molecular structure.

Project 3 extends Project 2:

| Project | Main representation | Main question |
|---|---|---|
| Project 2 | Human-readable molecular descriptors | Can molecular properties such as MolWt, LogP and TPSA predict logS? |
| Project 3 | Morgan/ECFP fingerprints | Can local molecular fragments and substructure patterns predict logS? |

Core workflow:

```text
SMILES → RDKit Mol → Morgan fingerprint matrix → QSAR regression model → predicted logS
```

Learning goals:

- Load and clean a real molecular dataset.
- Convert SMILES into RDKit molecule objects.
- Generate descriptor features and Morgan fingerprint features.
- Compare descriptor-only, fingerprint-only and combined feature sets.
- Train several QSAR regression models.
- Evaluate performance with MAE, RMSE and R².
- Diagnose model errors with residuals, chemical class tags and applicability-domain checks.
- Predict solubility for new molecules.


## Chapter 0 — Environment notes

Recommended packages:

```bash
conda install -c conda-forge rdkit pandas numpy matplotlib scikit-learn jupyter
```

This notebook uses the public DeepChem ESOL/Delaney CSV file as a dataset source. If internet access is disabled, download the CSV manually and place it at:

```text
data/delaney-processed.csv
```


In [ ]:
# ============================================================
# Chapter 1 — Imports and global settings
# ============================================================

from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display

from rdkit import Chem, DataStructs
from rdkit.Chem import Descriptors, Crippen, Lipinski, rdMolDescriptors, Draw
from rdkit.Chem.rdFingerprintGenerator import GetMorganGenerator
from rdkit.Chem.Scaffolds import MurckoScaffold

from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, GradientBoostingRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
DATA_DIR = Path("data")
OUTPUT_DIR = Path("outputs")
DATA_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)

print("Imports completed.")


## Chapter 2 — Load the ESOL / Delaney solubility dataset

The ESOL / Delaney dataset contains molecular structures and experimentally measured aqueous solubility values. The target column is log solubility in mols per litre.

In this notebook:

```text
target_logS = measured log solubility in mols per litre
```

Higher `logS` means more soluble. Lower `logS` means less soluble.


In [ ]:
# ============================================================
# Chapter 2A — Load ESOL / Delaney data
# ============================================================

ESOL_URL = "https://raw.githubusercontent.com/deepchem/deepchem/master/datasets/delaney-processed.csv"
LOCAL_ESOL_PATH = DATA_DIR / "delaney-processed.csv"

try:
    df_raw = pd.read_csv(ESOL_URL)
    print("Loaded ESOL dataset from URL.")
except Exception as exc:
    if LOCAL_ESOL_PATH.exists():
        df_raw = pd.read_csv(LOCAL_ESOL_PATH)
        print("Loaded ESOL dataset from local file:", LOCAL_ESOL_PATH)
    else:
        raise RuntimeError(
            "Could not load ESOL dataset from URL and local file was not found. "
            "Download delaney-processed.csv into data/."
        ) from exc

print("Raw shape:", df_raw.shape)
print("Raw columns:", df_raw.columns.tolist())
display(df_raw.head())


In [ ]:
# ============================================================
# Chapter 2B — Standardise columns for the QSAR pipeline
# ============================================================

required_columns = [
    "Compound ID",
    "smiles",
    "measured log solubility in mols per litre",
]

missing_columns = [col for col in required_columns if col not in df_raw.columns]
if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

df = df_raw.rename(columns={
    "Compound ID": "name",
    "measured log solubility in mols per litre": "target_logS",
})

df = df[["name", "smiles", "target_logS"]].dropna().copy()

df["name"] = df["name"].astype(str)
df["smiles"] = df["smiles"].astype(str)
df["target_logS"] = df["target_logS"].astype(float)

print("Prepared shape:", df.shape)
display(df.head())


## Chapter 3 — Convert SMILES to RDKit molecules

RDKit descriptors and fingerprints require RDKit `Mol` objects. Therefore, the first chemistry-processing step is:

```text
SMILES string → RDKit Mol object
```

This chapter also canonicalises SMILES and removes duplicate structures.


In [ ]:
# ============================================================
# Chapter 3A — SMILES validation and conversion
# ============================================================

def smiles_to_mol(smiles):
    """Convert a SMILES string into an RDKit Mol object. Return None if invalid."""
    if pd.isna(smiles):
        return None
    smiles = str(smiles).strip()
    if smiles == "":
        return None
    try:
        return Chem.MolFromSmiles(smiles)
    except Exception:
        return None


def mol_to_canonical_smiles(mol):
    """Return canonical RDKit SMILES for a valid molecule."""
    if mol is None:
        return None
    return Chem.MolToSmiles(mol, canonical=True)


df["mol"] = df["smiles"].apply(smiles_to_mol)
df["valid_mol"] = df["mol"].notna()

invalid_df = df.loc[~df["valid_mol"], ["name", "smiles", "target_logS"]]
print("Total molecules:", len(df))
print("Invalid molecules:", len(invalid_df))

display(invalid_df.head())


In [ ]:
# ============================================================
# Chapter 3B — Canonicalise and remove duplicate molecules
# ============================================================

df = df[df["valid_mol"]].copy()
df["canonical_smiles"] = df["mol"].apply(mol_to_canonical_smiles)

before_dedup = len(df)
df = df.drop_duplicates(subset="canonical_smiles").reset_index(drop=True)
after_dedup = len(df)

print("Duplicate structures removed:", before_dedup - after_dedup)
print("Clean dataset shape:", df.shape)

display(df[["name", "smiles", "canonical_smiles", "target_logS"]].head())


In [ ]:
# ============================================================
# Chapter 3C — Quick target distribution check
# ============================================================

print(df["target_logS"].describe())

plt.figure(figsize=(7, 4))
plt.hist(df["target_logS"], bins=30, edgecolor="black")
plt.xlabel("Measured logS")
plt.ylabel("Count")
plt.title("Distribution of ESOL logS target")
plt.tight_layout()
plt.show()


## Chapter 4 — Descriptor features for comparison

Project 3 focuses on Morgan fingerprints, but descriptor features are still useful as a comparison baseline.

These descriptors represent global molecular properties, including:

| Descriptor | Meaning |
|---|---|
| MolWt | Molecular size |
| LogP | Hydrophobicity / lipophilicity |
| TPSA | Polar surface area |
| HBD / HBA | Hydrogen bonding capacity |
| RotatableBonds | Molecular flexibility |
| RingCount | Ring-system complexity |
| FractionCSP3 | Saturation / 3D character |


In [ ]:
# ============================================================
# Chapter 4 — Calculate descriptor features
# ============================================================

def calculate_descriptors(mol):
    """Calculate a compact set of RDKit molecular descriptors."""
    return {
        "MolWt": Descriptors.MolWt(mol),
        "LogP": Crippen.MolLogP(mol),
        "TPSA": rdMolDescriptors.CalcTPSA(mol),
        "HBD": Lipinski.NumHDonors(mol),
        "HBA": Lipinski.NumHAcceptors(mol),
        "RotatableBonds": Lipinski.NumRotatableBonds(mol),
        "HeavyAtomCount": Lipinski.HeavyAtomCount(mol),
        "RingCount": Lipinski.RingCount(mol),
        "AromaticRingCount": rdMolDescriptors.CalcNumAromaticRings(mol),
        "AliphaticRingCount": rdMolDescriptors.CalcNumAliphaticRings(mol),
        "FractionCSP3": rdMolDescriptors.CalcFractionCSP3(mol),
        "BertzCT": Descriptors.BertzCT(mol),
        "BalabanJ": Descriptors.BalabanJ(mol),
        "LabuteASA": rdMolDescriptors.CalcLabuteASA(mol),
    }


desc_df = pd.DataFrame([calculate_descriptors(mol) for mol in df["mol"]])
desc_df = desc_df.replace([np.inf, -np.inf], np.nan)
desc_df = desc_df.fillna(desc_df.median(numeric_only=True))

print("Descriptor matrix shape:", desc_df.shape)
display(desc_df.head())


In [ ]:
# ============================================================
# Chapter 4B — Descriptor correlation with logS
# ============================================================

desc_with_target = desc_df.copy()
desc_with_target["target_logS"] = df["target_logS"].values

corr = (
    desc_with_target.corr(numeric_only=True)["target_logS"]
    .drop("target_logS")
    .sort_values(key=lambda s: s.abs(), ascending=False)
)

corr_df = corr.reset_index()
corr_df.columns = ["descriptor", "correlation_with_logS"]
display(corr_df)

plt.figure(figsize=(8, 5))
plt.barh(corr_df["descriptor"][::-1], corr_df["correlation_with_logS"][::-1])
plt.xlabel("Pearson correlation with logS")
plt.title("Descriptor correlation with solubility")
plt.tight_layout()
plt.show()


## Chapter 5 — Morgan fingerprints

Morgan fingerprints, also called ECFP-like fingerprints, encode local circular atom environments as fixed-length bit vectors.

Core idea:

```text
molecule → many local substructure environments → hashed fingerprint bits
```

Important parameters:

| Parameter | Meaning |
|---|---|
| `radius=2` | Atom environments up to two bond steps; commonly similar to ECFP4 |
| `fpSize=2048` | Length of the fingerprint vector |
| bit value `1` | At least one substructure was hashed to this bit |
| bit value `0` | No detected substructure for that bit |


In [ ]:
# ============================================================
# Chapter 5A — Generate Morgan fingerprints
# ============================================================

FP_RADIUS = 2
FP_SIZE = 2048

morgan_generator = GetMorganGenerator(radius=FP_RADIUS, fpSize=FP_SIZE)


def mol_to_morgan_fp(mol):
    """Convert an RDKit Mol into a dense NumPy Morgan fingerprint vector."""
    fp = morgan_generator.GetFingerprint(mol)
    return np.asarray(fp, dtype=np.float32)


fp_array = np.stack(df["mol"].apply(mol_to_morgan_fp).values)
fp_df = pd.DataFrame(fp_array, columns=[f"Morgan_{i}" for i in range(FP_SIZE)])

print("Morgan fingerprint matrix shape:", fp_df.shape)
display(fp_df.head())


In [ ]:
# ============================================================
# Chapter 5B — Fingerprint density analysis
# ============================================================

active_bits_per_molecule = fp_array.sum(axis=1)
bit_frequency = fp_array.sum(axis=0)

print("Active bits per molecule:")
print(pd.Series(active_bits_per_molecule).describe())

print("\nNumber of never-active bits:", int((bit_frequency == 0).sum()))
print("Number of active-at-least-once bits:", int((bit_frequency > 0).sum()))

plt.figure(figsize=(7, 4))
plt.hist(active_bits_per_molecule, bins=30, edgecolor="black")
plt.xlabel("Number of active Morgan bits per molecule")
plt.ylabel("Count")
plt.title("Fingerprint density per molecule")
plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
# Chapter 5C — Remove never-active fingerprint bits
# ============================================================

active_bit_mask = bit_frequency > 0
fp_reduced_array = fp_array[:, active_bit_mask]
active_bit_names = [f"Morgan_{i}" for i, is_active in enumerate(active_bit_mask) if is_active]
fp_reduced_df = pd.DataFrame(fp_reduced_array, columns=active_bit_names)

print("Original fingerprint shape:", fp_df.shape)
print("Reduced fingerprint shape:", fp_reduced_df.shape)


## Chapter 6 — Build feature sets

We compare three molecular representations:

| Feature set | Meaning |
|---|---|
| Descriptors only | Global molecular properties |
| Morgan only | Local substructure fingerprints |
| Descriptors + Morgan | Combined global + local structure representation |


In [ ]:
# ============================================================
# Chapter 6 — Build feature matrices and target vector
# ============================================================

X_desc = desc_df.reset_index(drop=True)
X_fp = fp_reduced_df.reset_index(drop=True)
X_combined = pd.concat([X_desc, X_fp], axis=1)

y = df["target_logS"].values.astype(float)

feature_sets = {
    "Descriptors only": X_desc,
    "Morgan only": X_fp,
    "Descriptors + Morgan": X_combined,
}

for feature_set_name, X in feature_sets.items():
    print(f"{feature_set_name}: X shape = {X.shape}")

print("Target shape:", y.shape)


## Chapter 7 — Model setup

Regression metrics used in this notebook:

| Metric | Interpretation |
|---|---|
| MAE | Average absolute prediction error |
| RMSE | Penalises large errors more strongly than MAE |
| R² | Fraction of variance explained by the model |

For `logS`, RMSE is useful because a 1-unit error means a large difference in solubility on the original scale.


In [ ]:
# ============================================================
# Chapter 7A — Regression metrics and model dictionary
# ============================================================

def rmse_score(y_true, y_pred):
    """Root mean squared error."""
    return mean_squared_error(y_true, y_pred) ** 0.5


def regression_metric_dict(y_true, y_pred):
    """Return common regression metrics as a dictionary."""
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": rmse_score(y_true, y_pred),
        "R2": r2_score(y_true, y_pred),
    }


def get_models():
    """Return regression models used for QSAR comparison."""
    return {
        "Linear Regression": Pipeline([
            ("scaler", StandardScaler()),
            ("model", LinearRegression())
        ]),
        "Ridge": Pipeline([
            ("scaler", StandardScaler()),
            ("model", Ridge(alpha=1.0))
        ]),
        "Lasso": Pipeline([
            ("scaler", StandardScaler()),
            ("model", Lasso(alpha=0.001, max_iter=10000))
        ]),
        "ElasticNet": Pipeline([
            ("scaler", StandardScaler()),
            ("model", ElasticNet(alpha=0.001, l1_ratio=0.5, max_iter=10000))
        ]),
        "Random Forest": RandomForestRegressor(
            n_estimators=400,
            random_state=RANDOM_STATE,
            n_jobs=-1
        ),
        "Extra Trees": ExtraTreesRegressor(
            n_estimators=400,
            random_state=RANDOM_STATE,
            n_jobs=-1
        ),
        "Gradient Boosting": GradientBoostingRegressor(
            random_state=RANDOM_STATE
        ),
        "SVR RBF": Pipeline([
            ("scaler", StandardScaler()),
            ("model", SVR(kernel="rbf", C=10.0, epsilon=0.1))
        ]),
        "kNN": Pipeline([
            ("scaler", StandardScaler()),
            ("model", KNeighborsRegressor(n_neighbors=5))
        ]),
    }


In [ ]:
# ============================================================
# Chapter 7B — Train/test evaluation helper
# ============================================================

def evaluate_train_test(X, y, feature_set_name, test_size=0.20, random_state=RANDOM_STATE):
    """Train all models on one feature set and evaluate on a holdout test split."""
    indices = np.arange(len(y))
    X_train, X_test, y_train, y_test, idx_train, idx_test = train_test_split(
        X,
        y,
        indices,
        test_size=test_size,
        random_state=random_state,
    )

    rows = []
    fitted_models = {}
    predictions = {}

    for model_name, model in get_models().items():
        model.fit(X_train, y_train)

        pred_train = model.predict(X_train)
        pred_test = model.predict(X_test)

        train_metrics = regression_metric_dict(y_train, pred_train)
        test_metrics = regression_metric_dict(y_test, pred_test)

        rows.append({
            "feature_set": feature_set_name,
            "model": model_name,
            "train_MAE": train_metrics["MAE"],
            "train_RMSE": train_metrics["RMSE"],
            "train_R2": train_metrics["R2"],
            "test_MAE": test_metrics["MAE"],
            "test_RMSE": test_metrics["RMSE"],
            "test_R2": test_metrics["R2"],
        })

        fitted_models[model_name] = model
        predictions[model_name] = pred_test

    result_df = pd.DataFrame(rows).sort_values("test_RMSE").reset_index(drop=True)

    split_data = {
        "X_train": X_train,
        "X_test": X_test,
        "y_train": y_train,
        "y_test": y_test,
        "idx_train": idx_train,
        "idx_test": idx_test,
    }

    return result_df, fitted_models, predictions, split_data


## Chapter 8 — Train/test model comparison

This chapter trains every model on every feature set. The result table allows you to ask:

- Do Morgan fingerprints improve over global descriptors?
- Does combining descriptors and Morgan bits help?
- Which model generalises best on the holdout test split?


In [ ]:
# ============================================================
# Chapter 8 — Run train/test comparison
# ============================================================

all_results = []
all_fitted_models = {}
all_predictions = {}
all_split_data = {}

for feature_set_name, X in feature_sets.items():
    result_df, fitted_models, predictions, split_data = evaluate_train_test(
        X,
        y,
        feature_set_name=feature_set_name,
    )

    all_results.append(result_df)
    all_fitted_models[feature_set_name] = fitted_models
    all_predictions[feature_set_name] = predictions
    all_split_data[feature_set_name] = split_data

results_df = (
    pd.concat(all_results, axis=0)
    .sort_values("test_RMSE")
    .reset_index(drop=True)
)

display(results_df)


In [ ]:
# ============================================================
# Chapter 8B — Visualise model comparison
# ============================================================

plot_df = results_df.copy()
plot_df["label"] = plot_df["feature_set"] + " | " + plot_df["model"]
plot_df = plot_df.sort_values("test_RMSE", ascending=True).head(15)

plt.figure(figsize=(9, 6))
plt.barh(plot_df["label"][::-1], plot_df["test_RMSE"][::-1])
plt.xlabel("Test RMSE")
plt.title("Top 15 train/test QSAR models")
plt.tight_layout()
plt.show()


## Chapter 9 — Cross-validation

A single train/test split can be noisy. Cross-validation gives a more stable estimate of model performance.

For this practice notebook, we use shuffled 5-fold cross-validation. In more rigorous chemistry benchmarking, scaffold split or time split may be more realistic.


In [ ]:
# ============================================================
# Chapter 9A — Cross-validation helper
# ============================================================

def evaluate_cv(X, y, feature_set_name, n_splits=5, random_state=RANDOM_STATE):
    """Evaluate all models with K-fold cross-validation."""
    cv = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)

    rows = []
    for model_name, model in get_models().items():
        scores = cross_val_score(
            model,
            X,
            y,
            scoring="neg_root_mean_squared_error",
            cv=cv,
            n_jobs=-1,
        )

        rmse_scores = -scores
        rows.append({
            "feature_set": feature_set_name,
            "model": model_name,
            "CV_RMSE_mean": rmse_scores.mean(),
            "CV_RMSE_std": rmse_scores.std(),
        })

    return pd.DataFrame(rows).sort_values("CV_RMSE_mean").reset_index(drop=True)


In [ ]:
# ============================================================
# Chapter 9B — Run cross-validation comparison
# ============================================================

all_cv_results = []

for feature_set_name, X in feature_sets.items():
    cv_df = evaluate_cv(X, y, feature_set_name=feature_set_name)
    all_cv_results.append(cv_df)

cv_results_df = (
    pd.concat(all_cv_results, axis=0)
    .sort_values("CV_RMSE_mean")
    .reset_index(drop=True)
)

display(cv_results_df)


In [ ]:
# ============================================================
# Chapter 9C — Select best model by CV RMSE
# ============================================================

best_row = cv_results_df.iloc[0]

best_feature_set_name = best_row["feature_set"]
best_model_name = best_row["model"]

print("Best feature set:", best_feature_set_name)
print("Best model:", best_model_name)
print("CV RMSE mean:", best_row["CV_RMSE_mean"])
print("CV RMSE std:", best_row["CV_RMSE_std"])


## Chapter 10 — Refit best model and run diagnostics

The next cells refit the selected model and inspect its behaviour using plots and error tables.


In [ ]:
# ============================================================
# Chapter 10A — Refit best model on the selected feature set
# ============================================================

X_best = feature_sets[best_feature_set_name]

best_result_df, fitted_models, predictions, split_data = evaluate_train_test(
    X_best,
    y,
    feature_set_name=best_feature_set_name,
)

best_model = get_models()[best_model_name]

X_train = split_data["X_train"]
X_test = split_data["X_test"]
y_train = split_data["y_train"]
y_test = split_data["y_test"]
idx_train = split_data["idx_train"]
idx_test = split_data["idx_test"]

best_model.fit(X_train, y_train)
y_pred = best_model.predict(X_test)

print("Best feature set:", best_feature_set_name)
print("Best model:", best_model_name)
print("Holdout metrics:")
print("MAE:", mean_absolute_error(y_test, y_pred))
print("RMSE:", rmse_score(y_test, y_pred))
print("R2:", r2_score(y_test, y_pred))


In [ ]:
# ============================================================
# Chapter 10B — Predicted vs actual plot
# ============================================================

plt.figure(figsize=(6, 6))
plt.scatter(y_test, y_pred, alpha=0.7)

min_val = min(y_test.min(), y_pred.min())
max_val = max(y_test.max(), y_pred.max())
plt.plot([min_val, max_val], [min_val, max_val], linestyle="--")

plt.xlabel("Actual logS")
plt.ylabel("Predicted logS")
plt.title(f"Actual vs Predicted logS\n{best_feature_set_name} | {best_model_name}")
plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
# Chapter 10C — Residual diagnostics
# ============================================================

residuals = y_pred - y_test

plt.figure(figsize=(7, 5))
plt.scatter(y_pred, residuals, alpha=0.7)
plt.axhline(0, linestyle="--")

plt.xlabel("Predicted logS")
plt.ylabel("Residual = predicted - actual")
plt.title(f"Residual Plot\n{best_feature_set_name} | {best_model_name}")
plt.tight_layout()
plt.show()

plt.figure(figsize=(7, 4))
plt.hist(residuals, bins=30, edgecolor="black")
plt.xlabel("Residual = predicted - actual")
plt.ylabel("Count")
plt.title("Residual distribution")
plt.tight_layout()
plt.show()


## Chapter 11 — Error analysis

Good QSAR notebooks should not stop at one metric. We should ask which molecules are badly predicted and why.

Possible reasons for large errors:

- The molecule has a chemical scaffold poorly represented in the training set.
- The molecule belongs to a chemical class with unusual solubility behaviour.
- The model relies on fingerprint similarity but the target depends on global properties.
- The experimental target may contain noise.


In [ ]:
# ============================================================
# Chapter 11A — Top prediction errors
# ============================================================

error_df = df.iloc[idx_test].copy()
error_df["actual_logS"] = y_test
error_df["predicted_logS"] = y_pred
error_df["error"] = error_df["predicted_logS"] - error_df["actual_logS"]
error_df["absolute_error"] = error_df["error"].abs()

error_df = error_df.sort_values("absolute_error", ascending=False).reset_index(drop=True)

display(error_df[["name", "smiles", "actual_logS", "predicted_logS", "error", "absolute_error"]].head(20))


In [ ]:
# ============================================================
# Chapter 11B — Draw top error molecules
# ============================================================

top_error_mols = error_df.head(12)["mol"].tolist()
top_error_labels = [
    f"{row.name}\nActual: {row.actual_logS:.2f}\nPred: {row.predicted_logS:.2f}\nAbs err: {row.absolute_error:.2f}"
    for row in error_df.head(12).itertuples()
]

img = Draw.MolsToGridImage(
    top_error_mols,
    molsPerRow=4,
    subImgSize=(260, 180),
    legends=top_error_labels,
)

display(img)


In [ ]:
# ============================================================
# Chapter 11C — Simple chemical class tagging
# ============================================================

def count_atom(mol, atomic_num):
    """Count atoms with a specific atomic number."""
    return sum(1 for atom in mol.GetAtoms() if atom.GetAtomicNum() == atomic_num)


def assign_simple_chemical_class(mol):
    """Assign a rough chemistry class for error analysis."""
    n_cl = count_atom(mol, 17)
    n_f = count_atom(mol, 9)
    n_br = count_atom(mol, 35)
    n_i = count_atom(mol, 53)
    n_p = count_atom(mol, 15)
    n_o = count_atom(mol, 8)
    n_n = count_atom(mol, 7)

    n_halogen = n_cl + n_f + n_br + n_i
    mol_logp = Crippen.MolLogP(mol)
    mol_wt = Descriptors.MolWt(mol)
    ring_count = Lipinski.RingCount(mol)
    aromatic_rings = rdMolDescriptors.CalcNumAromaticRings(mol)

    if n_halogen >= 3:
        return "highly halogenated"
    if n_p >= 1:
        return "phosphorus-containing"
    if mol_logp >= 5:
        return "high-logP hydrophobic"
    if mol_wt >= 400:
        return "large molecule"
    if aromatic_rings >= 2:
        return "polyaromatic"
    if n_o >= 2 or n_n >= 2:
        return "polar heteroatom-rich"
    if ring_count == 0 and mol_logp >= 3:
        return "acyclic hydrophobic"
    return "other"


df["simple_chemical_class"] = df["mol"].apply(assign_simple_chemical_class)
error_df = df.iloc[idx_test].copy()
error_df["actual_logS"] = y_test
error_df["predicted_logS"] = y_pred
error_df["error"] = error_df["predicted_logS"] - error_df["actual_logS"]
error_df["absolute_error"] = error_df["error"].abs()

display(error_df[["name", "simple_chemical_class", "actual_logS", "predicted_logS", "absolute_error"]].sort_values("absolute_error", ascending=False).head(20))


In [ ]:
# ============================================================
# Chapter 11D — Error summary by chemical class
# ============================================================

class_error_summary = (
    error_df
    .groupby("simple_chemical_class")
    .agg(
        n=("name", "count"),
        mean_abs_error=("absolute_error", "mean"),
        median_abs_error=("absolute_error", "median"),
        max_abs_error=("absolute_error", "max"),
        mean_error=("error", "mean"),
    )
    .sort_values("mean_abs_error", ascending=False)
)

display(class_error_summary)


## Chapter 12 — Applicability domain with Tanimoto similarity

QSAR predictions are more trustworthy when the test molecule is similar to molecules seen during training.

Here we estimate a simple applicability-domain score:

```text
for each test molecule: maximum Tanimoto similarity to any training molecule
```

Low similarity plus high error suggests the model is extrapolating beyond its reliable chemical domain.


In [ ]:
# ============================================================
# Chapter 12A — Compute maximum similarity to training set
# ============================================================

def mol_to_morgan_bitvect(mol):
    """Return RDKit explicit bit vector for Tanimoto similarity."""
    return morgan_generator.GetFingerprint(mol)


train_mols = df.iloc[idx_train]["mol"].tolist()
test_mols = df.iloc[idx_test]["mol"].tolist()

train_fps = [mol_to_morgan_bitvect(mol) for mol in train_mols]
test_fps = [mol_to_morgan_bitvect(mol) for mol in test_mols]

max_train_similarities = []
for test_fp in test_fps:
    similarities = DataStructs.BulkTanimotoSimilarity(test_fp, train_fps)
    max_train_similarities.append(max(similarities))

error_df["max_train_similarity"] = max_train_similarities

display(error_df[[
    "name",
    "simple_chemical_class",
    "actual_logS",
    "predicted_logS",
    "absolute_error",
    "max_train_similarity",
]].sort_values("absolute_error", ascending=False).head(20))


In [ ]:
# ============================================================
# Chapter 12B — Error vs similarity plot
# ============================================================

plt.figure(figsize=(7, 5))
plt.scatter(error_df["max_train_similarity"], error_df["absolute_error"], alpha=0.7)
plt.xlabel("Maximum Tanimoto similarity to training set")
plt.ylabel("Absolute error")
plt.title("Applicability domain: error vs training-set similarity")
plt.tight_layout()
plt.show()


## Chapter 13 — Scaffold split challenge

Random splits can overestimate QSAR performance because similar molecules can appear in both train and test sets. A scaffold split is harder: molecules with related core structures are kept together.

This gives a better estimate of whether the model can generalise to new chemical series.


In [ ]:
# ============================================================
# Chapter 13A — Generate Murcko scaffolds
# ============================================================

def get_murcko_scaffold(mol):
    """Return Bemis-Murcko scaffold SMILES for a molecule."""
    try:
        scaffold = MurckoScaffold.MurckoScaffoldSmiles(mol=mol)
        if scaffold is None:
            return ""
        return scaffold
    except Exception:
        return ""


df["scaffold"] = df["mol"].apply(get_murcko_scaffold)

print("Unique scaffolds:", df["scaffold"].nunique())
display(df[["name", "smiles", "scaffold"]].head())


In [ ]:
# ============================================================
# Chapter 13B — Create scaffold-based train/test indices
# ============================================================

def scaffold_train_test_indices(scaffold_series, test_fraction=0.20, random_state=RANDOM_STATE):
    """Create train/test indices by assigning whole scaffolds to one split."""
    rng = np.random.default_rng(random_state)

    scaffold_to_indices = {}
    for idx, scaffold in scaffold_series.items():
        scaffold_to_indices.setdefault(scaffold, []).append(idx)

    scaffold_groups = list(scaffold_to_indices.values())
    rng.shuffle(scaffold_groups)

    n_total = len(scaffold_series)
    n_test_target = int(round(n_total * test_fraction))

    test_indices = []
    train_indices = []

    for group in scaffold_groups:
        if len(test_indices) < n_test_target:
            test_indices.extend(group)
        else:
            train_indices.extend(group)

    return np.array(train_indices), np.array(test_indices)


scaffold_train_idx, scaffold_test_idx = scaffold_train_test_indices(df["scaffold"], test_fraction=0.20)

print("Scaffold train size:", len(scaffold_train_idx))
print("Scaffold test size:", len(scaffold_test_idx))


In [ ]:
# ============================================================
# Chapter 13C — Evaluate best model under scaffold split
# ============================================================

def evaluate_fixed_split(model, X, y, train_idx, test_idx):
    """Fit and evaluate one model using fixed train/test indices."""
    X_train_fixed = X.iloc[train_idx]
    X_test_fixed = X.iloc[test_idx]
    y_train_fixed = y[train_idx]
    y_test_fixed = y[test_idx]

    model.fit(X_train_fixed, y_train_fixed)
    pred_test_fixed = model.predict(X_test_fixed)

    return regression_metric_dict(y_test_fixed, pred_test_fixed), pred_test_fixed, y_test_fixed


scaffold_model = get_models()[best_model_name]
scaffold_metrics, scaffold_pred, scaffold_y_test = evaluate_fixed_split(
    scaffold_model,
    X_best,
    y,
    scaffold_train_idx,
    scaffold_test_idx,
)

print("Scaffold split metrics for selected model:")
print(scaffold_metrics)


## Chapter 14 — Model interpretation

Tree-based models provide feature importance. For fingerprint models, important features are often Morgan bits rather than human-readable descriptors. For combined feature sets, descriptor importance can still reveal whether global properties such as LogP and TPSA are influential.


In [ ]:
# ============================================================
# Chapter 14 — Feature importance or coefficient inspection
# ============================================================

def get_final_estimator(model):
    """Extract final estimator from a Pipeline or return model itself."""
    if hasattr(model, "named_steps") and "model" in model.named_steps:
        return model.named_steps["model"]
    return model


estimator = get_final_estimator(best_model)

if hasattr(estimator, "feature_importances_"):
    importances = estimator.feature_importances_
    importance_df = pd.DataFrame({
        "feature": X_best.columns,
        "importance": importances,
    }).sort_values("importance", ascending=False)

    display(importance_df.head(30))

    top_imp = importance_df.head(25).iloc[::-1]
    plt.figure(figsize=(8, 7))
    plt.barh(top_imp["feature"], top_imp["importance"])
    plt.xlabel("Feature importance")
    plt.title(f"Top feature importances: {best_model_name}")
    plt.tight_layout()
    plt.show()

elif hasattr(estimator, "coef_"):
    coefs = np.ravel(estimator.coef_)
    coef_df = pd.DataFrame({
        "feature": X_best.columns,
        "coefficient": coefs,
        "abs_coefficient": np.abs(coefs),
    }).sort_values("abs_coefficient", ascending=False)

    display(coef_df.head(30))

else:
    print("This model does not expose simple feature importances or coefficients.")


## Chapter 15 — Predict new molecules

This final prediction cell demonstrates how to apply the trained QSAR model to new SMILES strings.

Important limitation: the model predicts **solubility (`logS`)**, not odour, toxicity or fragrance quality.


In [ ]:
# ============================================================
# Chapter 15A — Featurise new molecules with matching columns
# ============================================================

def featurise_new_molecules(new_df):
    """Create descriptor, Morgan and combined features for new molecules."""
    new_df = new_df.copy()
    new_df["mol"] = new_df["smiles"].apply(smiles_to_mol)
    new_df = new_df[new_df["mol"].notna()].reset_index(drop=True)
    new_df["canonical_smiles"] = new_df["mol"].apply(mol_to_canonical_smiles)

    new_desc = pd.DataFrame([calculate_descriptors(mol) for mol in new_df["mol"]])
    new_desc = new_desc.replace([np.inf, -np.inf], np.nan)
    new_desc = new_desc.fillna(desc_df.median(numeric_only=True))

    new_fp_all = np.stack(new_df["mol"].apply(mol_to_morgan_fp).values)
    new_fp_reduced = new_fp_all[:, active_bit_mask]
    new_fp = pd.DataFrame(new_fp_reduced, columns=active_bit_names)

    if best_feature_set_name == "Descriptors only":
        X_new = new_desc[X_desc.columns]
    elif best_feature_set_name == "Morgan only":
        X_new = new_fp[X_fp.columns]
    else:
        X_new = pd.concat([new_desc[X_desc.columns], new_fp[X_fp.columns]], axis=1)
        X_new = X_new[X_combined.columns]

    return new_df, X_new


In [ ]:
# ============================================================
# Chapter 15B — Predict logS for example molecules
# ============================================================

new_molecules = pd.DataFrame({
    "name": [
        "benzyl acetate",
        "glucose",
        "ambroxide",
        "limonene",
        "vanillin",
        "caffeine",
    ],
    "smiles": [
        "CC(=O)OCC1=CC=CC=C1",
        "C(C1C(C(C(C(O1)O)O)O)O)O",
        "CC1(C)CCCC2(C)C1CCC(O2)C(C)(C)O",
        "CC1=CCC(CC1)C(=C)C",
        "COC1=C(C=CC(=C1)C=O)O",
        "Cn1cnc2c1c(=O)n(C)c(=O)n2C",
    ],
})

new_molecules_clean, X_new = featurise_new_molecules(new_molecules)

final_model = get_models()[best_model_name]
final_model.fit(X_train, y_train)

new_molecules_clean["predicted_logS"] = final_model.predict(X_new)

display(new_molecules_clean[["name", "smiles", "canonical_smiles", "predicted_logS"]])


In [ ]:
# ============================================================
# Chapter 15C — Draw predicted molecules
# ============================================================

pred_mols = new_molecules_clean["mol"].tolist()
pred_labels = [
    f"{row.name}\nPred logS: {row.predicted_logS:.2f}"
    for row in new_molecules_clean.itertuples()
]

img = Draw.MolsToGridImage(
    pred_mols,
    molsPerRow=3,
    subImgSize=(260, 180),
    legends=pred_labels,
)

display(img)


## Chapter 16 — Save outputs

The saved CSV files are useful for GitHub documentation and later project comparison.


In [ ]:
# ============================================================
# Chapter 16 — Save model comparison and prediction outputs
# ============================================================

results_path = OUTPUT_DIR / "project3_train_test_results.csv"
cv_path = OUTPUT_DIR / "project3_cv_results.csv"
errors_path = OUTPUT_DIR / "project3_test_set_errors.csv"
new_predictions_path = OUTPUT_DIR / "project3_new_molecule_predictions.csv"

results_df.to_csv(results_path, index=False)
cv_results_df.to_csv(cv_path, index=False)
error_df.to_csv(errors_path, index=False)
new_molecules_clean[["name", "smiles", "canonical_smiles", "predicted_logS"]].to_csv(new_predictions_path, index=False)

print("Saved:")
print(results_path)
print(cv_path)
print(errors_path)
print(new_predictions_path)


## Chapter 17 — Final interpretation guide

Use these rules when writing the Project 3 README summary:

1. **Morgan fingerprints are substructure features.** They represent local atom environments rather than direct physicochemical descriptors.
2. **Descriptors and fingerprints answer different questions.** Descriptors describe global molecular properties; fingerprints describe structural fragments.
3. **A lower RMSE means better solubility prediction.** Compare train, test and CV RMSE rather than relying only on one split.
4. **Overfitting is visible when train RMSE is much lower than test RMSE.** Tree ensembles can memorise fingerprint patterns if the dataset is small.
5. **Applicability domain matters.** A prediction is more trustworthy if the new molecule is similar to the training chemistry.
6. **Scaffold split is harder than random split.** Performance usually drops because test molecules contain less familiar core structures.

One-line summary:

```text
Project 3 uses Morgan fingerprints to turn molecular substructures into numerical vectors for QSAR regression, then compares descriptor-only, fingerprint-only and combined representations for logS prediction.
```
